# Minimal working example using BERT (`distBERT` model)

In [1]:
!pip install pandas torch transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.0 MB/s eta 0:00:00


## Setup

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

## Pseudo data creation

In [3]:
# # ----------------------------
# # 1) Sample pseudo "data"
# # ----------------------------
# items = pd.DataFrame([
#     {"item_id":"i1","title":"Noise Cancelling Headphones","description":"Wireless noise-cancelling headphones with 30-hour battery life","category":"electronics"},
#     {"item_id":"i2","title":"Mechanical Keyboard","description":"Mechanical keyboard with RGB and hot-swappable switches","category":"electronics"},
#     {"item_id":"i3","title":"Running Shoes","description":"Running shoes designed for long distance comfort and stability","category":"sports"},
#     {"item_id":"i4","title":"Vegetarian Cookbook","description":"Cookbook featuring quick vegetarian recipes for busy weeknights","category":"books"},
#     {"item_id":"i5","title":"Fitness Smartwatch","description":"Smartwatch with heart rate monitoring, GPS, and sleep tracking","category":"electronics"},
# ])

# events = pd.DataFrame([
#     {"user_id":"u1","item_id":"i1","ts":"2026-01-10"},
#     {"user_id":"u1","item_id":"i5","ts":"2026-01-11"},
#     {"user_id":"u2","item_id":"i2","ts":"2026-01-12"},
#     {"user_id":"u3","item_id":"i3","ts":"2026-01-13"},
#     {"user_id":"u3","item_id":"i4","ts":"2026-01-14"},
# ])
# events["ts"] = pd.to_datetime(events["ts"])



items = pd.read_csv("items.csv")
events = pd.read_csv("events.csv")
events["ts"] = pd.to_datetime(events["ts"])

print("Items shape:", items.shape)
print(items.head())
print("\nEvents shape:", events.shape)
print(events.head())


Items shape: (20, 7)
  item_id                        title  \
0      i1  Noise Cancelling Headphones   
1      i2          Mechanical Keyboard   
2      i3                Running Shoes   
3      i4          Vegetarian Cookbook   
4      i5           Fitness Smartwatch   

                                         description     category  \
0  Wireless noise-cancelling headphones with 30-h...  electronics   
1  Mechanical keyboard with RGB backlighting, hot...  electronics   
2  Running shoes designed for long distance comfo...       sports   
3  Cookbook featuring quick vegetarian recipes, p...        books   
4  Smartwatch with heart rate monitoring, GPS, sl...  electronics   

             brand   price                               tags  
0        SoundPeak  199.99      audio,wireless,travel,premium  
1         KeyForge  129.00  keyboard,gaming,productivity,desk  
2        StrideLab  110.00   running,fitness,outdoors,comfort  
3  HomeTable Press   24.99    cooking,vegetarian,recipe

## BERT encoder (What makes BERT work?)

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Evaluate the BERT encoder**

In [5]:
# ----------------------------
# 2) BERT encode helper
# ----------------------------

# turn off gradient calculation (no back propagation in using BERT)
@torch.no_grad()
def bert_embed(texts, max_len=256): # double or triple max_len
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)


## Embedding items into vectors (for comparisons)

In [6]:
# ----------------------------
# 3) Offline job: item embeddings
# ----------------------------

items["tags_text"] = items["tags"].str.replace(",", " ", regex=False)

items["text"] = (
    items["title"] + ". "
    + items["description"] + " "
    + "Category: " + items["category"] + ". "
    + "Brand: " + items["brand"] + ". "
    + "Tags: " + items["tags_text"]
)

item_vecs = bert_embed(items["text"].tolist())
item_id_list = items["item_id"].tolist()

# Build ANN index (inner product works with normalized vectors)
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)

print(f"Indexed {len(item_id_list)} items with embedding dim {item_vecs.shape[1]}")


Indexed 20 items with embedding dim 768


## What user-specific data are there?

In [7]:
# ----------------------------
# 4) Feature builder: user text from last N clicks
# ----------------------------
def build_user_text(user_id, events, items, N=8):
    hist = (events[events["user_id"] == user_id]
            .sort_values("ts")
            .tail(N)["item_id"]
            .tolist())
    if not hist:
        return "no history"
    text = items.set_index("item_id").loc[hist, "text"].tolist()
    return " ".join(text), set(hist)


## How to recommend "similar" item with BERT?

In [8]:
# ----------------------------
# 5) "What to recommend" function
# ----------------------------
def recommend(user_id, k=7): # mess with k
    user_text, seen = build_user_text(user_id, events, items, N=3)
    u = bert_embed([user_text])  # (1, 768)
    scores, idx = index.search(u, k + len(seen))  # keep track of what was seen by the user
    recs = []
    for j in idx[0]:
        iid = item_id_list[j]
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break
    return recs


In [10]:
for u in ["u1", "u2", "u3"]:
    print("=" * 60)
    print(f"USER: {u}")
    print("=" * 60)

    # event history: sorted by time, select readable columns
    user_events = (
        events[events["user_id"] == u]
        .sort_values("ts")
        [["ts", "item_id", "event_type", "dwell_seconds", "device"]]
        .reset_index(drop=True)
    )
    print("\nUser Events:")
    print(user_events.to_string(index=False))

    # recommendations
    rec_ids = recommend(u, k=3)
    print("\nRecommended Item IDs:", rec_ids)

    # full item details for recommended items
    rec_items = (
        items[items["item_id"].isin(rec_ids)]
        [["item_id", "title", "category", "brand", "price"]]
        .reset_index(drop=True)
    )
    print("\nRecommended Items:")
    print(rec_items.to_string(index=False))
    print()


USER: u1

User Events:
                 ts item_id  event_type  dwell_seconds  device
2026-01-01 07:47:00      i2       click             56  tablet
2026-01-04 08:01:00     i13        view              7  tablet
2026-01-07 14:28:00      i2       click             72  mobile
2026-01-07 19:06:00     i13        view             67  tablet
2026-01-10 08:42:00      i1 add_to_cart             46 desktop
2026-01-10 10:59:00      i7       click             43  tablet
2026-01-12 21:40:00      i6        view             59  tablet
2026-01-13 14:10:00     i17       click             99  tablet
2026-01-15 08:14:00     i11        view            105  mobile
2026-01-18 19:56:00     i15        view             77  mobile
2026-01-20 20:57:00      i2    purchase             98  mobile
2026-01-21 12:50:00      i3        view             77  tablet
2026-01-23 07:43:00     i14       click             82 desktop
2026-01-24 16:53:00      i6        view             57  tablet
2026-01-24 21:00:00      i9    p